# 17.10 多臂赌博机与上下文赌博机 / Bandits & Contextual Bandits

**中文**：Part 17 的收官。**赌博机(bandit)** 其实是**最简单的强化学习**——把它放进我们学过的 RL 谱系里看,一切豁然开朗:
**English**: The finale of Part 17. **Bandits** are actually the **simplest reinforcement learning** — placing them in the RL spectrum we've built clarifies everything:

| | 有状态? | 有状态转移? | 例子 |
|---|---|---|---|
| **多臂赌博机 MAB** | 否(单状态)| 否 | 拉哪个老虎机 / which slot machine |
| **上下文赌博机 Contextual** | 有(上下文)| 否(动作不改变环境)| 给**这个用户**推哪篇新闻 |
| **完整 RL (MDP)** | 有 | **有**(动作影响未来状态)| 下棋、机器人、游戏 |

**中文**：所以:**赌博机 = 无状态的一步 RL**;**上下文赌博机 = 有上下文但无状态转移的一步 RL**;**完整 RL = 多步、动作会改变未来**。三者共享同一个灵魂——**探索 vs 利用(exploration vs exploitation)**。本节从 RL 视角重看赌博机,并落到工业界最经典的应用:**新闻推荐**,还会讲一个前面没讲的关键实战问题——**离线(反事实)评估**。
**English**: So: **bandit = stateless one-step RL**; **contextual bandit = one-step RL with context but no state transitions**; **full RL = multi-step, where actions change the future**. All three share one soul — **exploration vs exploitation**. This section revisits bandits from the RL angle, lands on industry's classic application — **news recommendation** — and covers a key practical issue not seen before: **offline (counterfactual) evaluation**.

> **中文**：📎 我们在 **15.10(推荐系统)** 已从零实现过 ε-greedy / UCB / Thompson / LinUCB 的算法细节与遗憾理论。本节**不重复算法推导**,而是补上两块:①把赌博机**放进 RL 谱系**理解;②**离线评估**(工业推荐/RL 落地的头号难题)。
> **English**: 📎 In **15.10 (Recommender Systems)** we implemented ε-greedy / UCB / Thompson / LinUCB and their regret theory from scratch. This section **does not repeat the derivations**; instead it adds two things: ① placing bandits **in the RL spectrum**; ② **offline evaluation** (the #1 practical challenge in deploying recsys/RL).

---

> 💡 **面试速查 / Interview cheat-sheet（★★★ 推荐/RL 落地必考）**
> **中文**：**赌博机=一步 RL(无状态转移)**——所以"未来价值"退化为"即时奖励", 不需要贝尔曼递归, 是 RL 里唯一能拿到理论最优遗憾界的一类。**上下文赌博机**=每步给个特征 x, 最优臂随 x 变(LinUCB: 每臂在线岭回归+置信上界)。**vs 完整 RL**:赌博机的动作**不影响未来**(用户看没看这条新闻不改变下一个用户), 所以无需长期规划——很多"推荐/广告"问题本质是上下文赌博机而非完整 RL。**离线评估难题**:线上日志是**被某个策略筛选过的(有偏)**, 不能直接算新策略好坏; 解法:**重要性采样(IPS)/ 反事实 / replay**(需要 logging policy 的动作概率)。RLHF 也是"bandit 式反馈"(对一整个回答给一个偏好分)。
> **English**: **Bandit = one-step RL (no state transitions)** — so "future value" collapses to "immediate reward," no Bellman recursion needed; it's the one RL class with provably optimal regret bounds. **Contextual bandit** = a feature x each step, the best arm varies with x (LinUCB: per-arm online ridge regression + upper confidence bound). **vs full RL**: a bandit's action **doesn't affect the future** (whether this user saw the article doesn't change the next user), so no long-term planning — many "recommendation/ads" problems are contextual bandits, not full RL. **Offline-evaluation challenge**: online logs are **filtered by some policy (biased)**, so you can't directly score a new policy; fixes: **importance sampling (IPS) / counterfactual / replay** (need the logging policy's action probabilities). RLHF is also "bandit-style feedback" (one preference score for a whole answer).


In [ ]:

# ============================================================
# 新闻推荐上下文赌博机模拟器 / news-recommendation contextual-bandit simulator
# 中文:每来一个用户带 d 维上下文 x(画像/时段/设备...)。有 K 篇候选新闻, 每篇有隐藏偏好 θ_a。
#      展示第 a 篇的点击概率 = sigmoid(x·θ_a) —— 最优新闻随用户上下文而变。
#      智能体每步选一篇展示, 只观测到"被展示那篇"的点击/不点击(bandit 反馈)。目标:最大化累计点击率。
# English: each user arrives with d-dim context x (profile/time/device...). K candidate articles,
#      each with a hidden preference θ_a. Click prob of showing article a = sigmoid(x·θ_a) — the best
#      article varies with context. The agent shows one article per step and only observes the click
#      of the shown one (bandit feedback). Goal: maximize cumulative click-through rate (CTR).
# ============================================================
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(0)
d, K, T = 6, 10, 6000                                        # 上下文维度 / 新闻数 / 轮数
def make_world(seed): return np.random.default_rng(seed).normal(0,1,(K,d))   # 每篇新闻的隐藏偏好 θ

def run_bandit(algo, alpha=1.0, eps=0.1, trials=15):
    ctr_curve=np.zeros(T)
    for tr in range(trials):
        Theta=make_world(tr)                                 # 本次实验的"世界" / this run's world
        A=[np.eye(d) for _ in range(K)]; b=[np.zeros(d) for _ in range(K)]  # LinUCB 每臂统计量
        clicks=0; cum=np.zeros(T)
        for t in range(T):
            x=rng.normal(0,1,d); x/=np.linalg.norm(x)         # 本轮用户上下文 / this user's context
            probs=1/(1+np.exp(-Theta@x))                      # 各新闻真实点击率(随 x 变)/ true CTRs
            if algo=="linucb":                                # LinUCB:估计均值+置信上界(乐观探索)
                score=[]
                for a in range(K):
                    Ai=np.linalg.inv(A[a]); th=Ai@b[a]
                    score.append(x@th + alpha*np.sqrt(x@Ai@x))
                a=int(np.argmax(score))
            elif algo=="egreedy":                             # ε-greedy:偶尔随机, 否则选估计最高
                if rng.random()<eps: a=int(rng.integers(K))
                else: a=int(np.argmax([x@(np.linalg.inv(A[k])@b[k]) for k in range(K)]))
            elif algo=="random": a=int(rng.integers(K))       # 随机基线 / random baseline
            elif algo=="oracle": a=int(np.argmax(probs))      # 上帝视角(上界)/ oracle upper bound
            r=1.0 if rng.random()<probs[a] else 0.0           # 观测点击 / observe click
            A[a]+=np.outer(x,x); b[a]+=r*x                    # 更新被选臂 / update chosen arm
            clicks+=r; cum[t]=clicks/(t+1)                    # 累计 CTR / running CTR
        ctr_curve+=cum
    return ctr_curve/trials

results={name:run_bandit(name) for name in ["random","egreedy","linucb","oracle"]}
print(f"{'策略/policy':<14}{'最终累计点击率 final CTR':>24}")
for name in ["random","egreedy","linucb","oracle"]:
    print(f"{name:<14}{results[name][-1]:>24.3f}")


**中文**：LinUCB(上下文 + 乐观探索)明显胜过 ε-greedy 和随机,逼近上帝视角(oracle)。可视化累计点击率曲线,并看**为什么上下文重要**——把 LinUCB 和一个"忽略上下文"的版本对比。
**English**: LinUCB (context + optimistic exploration) clearly beats ε-greedy and random, approaching the oracle. Visualize the cumulative-CTR curves and see **why context matters** — comparing LinUCB with a "context-blind" version.


In [ ]:

# ============================================================
# 忽略上下文的赌博机(只学每篇全局CTR)/ context-blind bandit + visualization
# ============================================================
def run_contextfree(trials=15):
    ctr_curve=np.zeros(T)
    for tr in range(trials):
        Theta=make_world(tr); n=np.zeros(K); s=np.zeros(K); clicks=0; cum=np.zeros(T)
        for t in range(T):
            x=rng.normal(0,1,d); x/=np.linalg.norm(x); probs=1/(1+np.exp(-Theta@x))
            a=int(np.argmax((s+1)/(n+2)))                     # 只按全局平均CTR选(忽略x)/ ignore context
            r=1.0 if rng.random()<probs[a] else 0.0; n[a]+=1; s[a]+=r; clicks+=r; cum[t]=clicks/(t+1)
        ctr_curve+=cum
    return ctr_curve/trials
cf=run_contextfree()

fig,ax=plt.subplots(1,2,figsize=(14,4.7))
cmap={"random":"#8C8C8C","egreedy":"#DD8452","linucb":"#4C72B0","oracle":"#000000"}
for name in ["oracle","linucb","egreedy","random"]:
    ax[0].plot(results[name],color=cmap[name],lw=2,ls=":" if name=="oracle" else "-",label=f"{name} ({results[name][-1]:.3f})")
ax[0].set_title("新闻推荐:累计点击率 / cumulative CTR"); ax[0].set_xlabel("用户到达 t"); ax[0].set_ylabel("累计 CTR"); ax[0].legend(fontsize=9)
ax[1].plot(results["linucb"],color="#4C72B0",lw=2,label=f"LinUCB 用上下文 ({results['linucb'][-1]:.3f})")
ax[1].plot(cf,color="#C44E52",lw=2,label=f"忽略上下文 ({cf[-1]:.3f})")
ax[1].plot(results["oracle"],color="k",ls=":",label="oracle")
ax[1].set_title("上下文的价值:最优新闻随用户变 / value of context"); ax[1].set_xlabel("用户到达 t"); ax[1].set_ylabel("累计 CTR"); ax[1].legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/rl10_viz.png",dpi=80); plt.show()
print(f"LinUCB(用上下文) {results['linucb'][-1]:.3f}  vs  忽略上下文 {cf[-1]:.3f} —— 个性化的价值")


**中文**：现在讲一个工业界的**头号实战难题——离线评估**。你训练了一个新推荐策略,想知道它好不好,但**不能直接上线**(万一很烂会损失真金白银和用户)。可你手里的历史日志是**旧策略筛选过的**(旧策略没展示的新闻,你根本不知道用户会不会点)——直接拿日志算新策略的 CTR 会**严重有偏**。
**English**: Now a top practical challenge — **offline evaluation**. You trained a new recommendation policy and want to know if it's good, but you **can't just deploy it** (if it's bad you lose real money and users). Yet your historical logs were **filtered by the old policy** (for articles the old policy never showed, you have no idea whether users would click) — directly computing the new policy's CTR from logs is **severely biased**.

**中文**：一个简单而强大的解法是 **Replay(重放)方法**(Li et al., 2010):如果日志是用**均匀随机**策略采集的,那么要评估任意新策略,只需扫过日志、**只保留"新策略选择的动作恰好等于当时随机展示的动作"的那些记录**,统计这些记录里的点击率——这是新策略 CTR 的**无偏估计**!下面用模拟验证:离线 replay 估出的 CTR ≈ 真实上线 CTR。
**English**: A simple yet powerful fix is the **Replay method** (Li et al., 2010): if the logs were collected by a **uniformly random** policy, then to evaluate any new policy, scan the log and **keep only records where the new policy's chosen action happens to equal the randomly-shown action**, and compute the CTR over those — this is an **unbiased estimate** of the new policy's CTR! We verify by simulation: the offline replay estimate ≈ the true online CTR.


In [ ]:

# ============================================================
# 离线评估:Replay 方法 / offline evaluation via the Replay method
# ============================================================
# 1) 用【均匀随机】logging 策略采集日志 / collect logs with a UNIFORM-RANDOM logging policy
Theta=make_world(123); log=[]
for t in range(200000):
    x=rng.normal(0,1,d); x/=np.linalg.norm(x)
    a=int(rng.integers(K))                                    # 随机展示 / random action (logging policy)
    r=1.0 if rng.random()<1/(1+np.exp(-Theta[a]@x)) else 0.0
    log.append((x,a,r))

# 2) 一个"待评估"的目标策略(这里用一个已知不错的线性策略)/ a target policy to evaluate offline
#    先在另一半在线数据上学个 LinUCB 的点估计当目标策略 / learn a linear policy as the target
A=[np.eye(d) for _ in range(K)]; b=[np.zeros(d) for _ in range(K)]
for x,a,r in log[:50000]:                                    # 用前一段日志拟合 / fit on part of the log
    A[a]+=np.outer(x,x); b[a]+=r*x
W=np.array([np.linalg.inv(A[k])@b[k] for k in range(K)])     # 每臂线性权重 / per-arm linear weights
def target_policy(x): return int(np.argmax(W@x))             # 目标策略:选估计CTR最高的新闻 / greedy target

# 3) Replay 离线评估:只在"目标选的=日志随机选的"处计数 / replay: match target action to logged action
match=0; clicks=0
for x,a,r in log[50000:]:                                    # 用剩下的日志评估 / evaluate on held-out log
    if target_policy(x)==a:                                  # 关键:动作匹配才计入 / count only when actions match
        match+=1; clicks+=r
offline_ctr=clicks/match

# 4) 真实上线该目标策略, 得到"真值" / actually deploy the target policy online for the ground truth
online_clicks=0; N=60000
for _ in range(N):
    x=rng.normal(0,1,d); x/=np.linalg.norm(x); a=target_policy(x)
    online_clicks += 1 if rng.random()<1/(1+np.exp(-Theta[a]@x)) else 0
online_ctr=online_clicks/N
print(f"离线 Replay 估计 CTR / offline replay estimate: {offline_ctr:.3f}  (用了 {match} 条匹配日志)")
print(f"真实上线 CTR      / true online CTR:           {online_ctr:.3f}")
print(f"误差 / error: {abs(offline_ctr-online_ctr):.3f}  → Replay 几乎无偏地评估了新策略, 无需上线!")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **赌博机是理解 RL 的最佳起点也是终点**:它剥掉了"状态转移/长期规划"的复杂度,只留下 RL 最纯粹的内核——**探索 vs 利用**。这里有个诚实且有力的现象:**"忽略上下文"版本只有 ~0.50、几乎等于随机**!因为在本模拟里**没有哪篇新闻是"全局最好"的**——最优新闻**完全取决于用户上下文**(点击率是 $\sigma(x\cdot\theta)$,对随机 x 平均后每篇都约 0.5)。所以只有 **用上下文的 LinUCB(~0.74) 才能逼近上帝视角(~0.80)**。这正是个性化的价值:当"最优选择因人而异"时,不看用户特征等于瞎猜。
2. **很多"推荐/广告/搜索"其实是上下文赌博机, 不是完整 RL**:因为展示这条新闻**不改变下一个用户的状态**(无状态转移)。想清楚这一点很重要——如果动作不影响未来,就别用复杂的完整 RL(样本效率低、难训),用上下文赌博机又简单又有理论保证。**只有当"当前动作会改变未来"(如推荐会影响用户长期留存/养成习惯)时,才需要完整 RL。**
3. **离线评估是落地的命门**:我们用 Replay 方法**不上线**就估准了新策略的 CTR(离线 ≈ 真实,误差极小)。这解决了工业界最痛的问题——线上实验昂贵且有风险,而日志有偏。代价:Replay 要求日志来自随机(或已知概率)策略,且**匹配率低时会浪费大量数据**(20万条日志只有部分匹配)。更高级的 IPS/DR(双重稳健)估计能缓解。
4. **RLHF 也是 bandit 式的**:对大模型的一整个回答给一个偏好分(而非每个 token 的奖励),本质是**上下文赌博机反馈**——这把整个 Part 17 和大模型对齐连了起来。

**English**:
1. **Bandits are the best entry and exit point for understanding RL**: stripping away "state transitions / long-term planning," they leave RL's purest core — **exploration vs exploitation**. An honest, striking phenomenon here: the **context-blind version scores only ~0.50, basically random**! Because in this simulation **no article is "globally best"** — the best article **depends entirely on the user's context** (click prob is $\sigma(x\cdot\theta)$, averaging ~0.5 per article over random x). So only **context-using LinUCB (~0.74) approaches the oracle (~0.80)**. That is the value of personalization: when "the best choice varies per person," ignoring user features is just guessing.
2. **Many "recommendation/ads/search" problems are contextual bandits, not full RL**: showing this article **doesn't change the next user's state** (no transitions). Recognizing this matters — if actions don't affect the future, don't use complex full RL (sample-inefficient, hard to train); a contextual bandit is simpler with theoretical guarantees. **Only when "the current action changes the future" (e.g. recommendations shaping long-term retention/habits) do you need full RL.**
3. **Offline evaluation is the crux of deployment**: with the Replay method we estimated the new policy's CTR accurately **without deploying** (offline ≈ true, tiny error). This solves industry's biggest pain — online experiments are expensive and risky, while logs are biased. The cost: Replay needs logs from a random (or known-probability) policy and **wastes lots of data when the match rate is low** (only part of 200k logs matched). More advanced IPS/DR (doubly-robust) estimators mitigate this.
4. **RLHF is also bandit-style**: one preference score for an entire LLM answer (not per-token rewards) is essentially **contextual-bandit feedback** — tying all of Part 17 to LLM alignment.

> 💼 **实战视角 / Practical angle**
> **中文**:上下文赌博机是**推荐/广告/个性化的工业主力**(Yahoo/今日头条的新闻推荐、广告出价、A/B 的智能版)。落地要点:①**先判断问题是不是赌博机**(动作影不影响未来)——是就别上完整 RL;②**离线评估(IPS/DR/Replay)** 是上线前的必经关卡;③探索要"有预算"(别为探索损失太多真实收益);④冷启动、反馈延迟(点击几天后才转化)、非平稳(热点变化)是真实系统的三大坑。面试金句:*"赌博机是无状态转移的一步 RL, 核心只有探索-利用; 上下文赌博机=LinUCB 那类, 很多推荐/广告本质是它而非完整 RL; 上线前用 IPS/Replay 做离线反事实评估。"*
> **English**: Contextual bandits are the **industrial workhorse of recommendation/ads/personalization** (Yahoo/Toutiao news recommendation, ad bidding, "smart A/B"). Deployment keys: ① **first decide whether the problem is a bandit** (do actions affect the future?) — if so, don't use full RL; ② **offline evaluation (IPS/DR/Replay)** is a mandatory gate before deployment; ③ exploration must be "budgeted" (don't lose too much real revenue exploring); ④ cold-start, delayed feedback (conversions days after the click), and non-stationarity (trends shift) are the three real-world pitfalls. Interview line: *"A bandit is one-step RL with no state transitions — only explore-exploit; contextual bandits (LinUCB-style) underlie much of recommendation/ads rather than full RL; before deploying, use IPS/Replay for offline counterfactual evaluation."*

---
### 小结 / Summary
- **中文**:赌博机=一步无状态转移 RL; 上下文赌博机=有上下文的一步 RL(LinUCB); 完整 RL=动作改变未来。
- **English**: Bandit = one-step RL with no transitions; contextual bandit = one-step RL with context (LinUCB); full RL = actions change the future.
- **中文**:新闻推荐上 LinUCB(用上下文+乐观探索)明显胜过忽略上下文与随机, 逼近 oracle。
- **English**: On news recommendation, LinUCB (context + optimistic exploration) clearly beats context-blind and random, approaching the oracle.
- **中文**:离线(反事实)评估(Replay/IPS)让你不上线就估准新策略——工业落地的命门。
- **English**: Offline (counterfactual) evaluation (Replay/IPS) estimates a new policy without deploying — the crux of real-world deployment.

---
**中文**：🎉 至此 **Part 17 · 强化学习** 全部完成!从 MDP/贝尔曼 → 动态规划 → 蒙特卡洛 → TD/SARSA → Q-learning → DQN → 策略梯度 → Actor-Critic → PPO → 赌博机,你已经**从零实现**并跑通了强化学习的完整主线,建立了"价值 vs 策略""有模型 vs 无模型""on-policy vs off-policy""探索 vs 利用"的核心直觉,并一路诚实地面对每个算法的真实表现。
**English**: 🎉 **Part 17 · Reinforcement Learning** is complete! From MDP/Bellman → dynamic programming → Monte Carlo → TD/SARSA → Q-learning → DQN → policy gradient → Actor-Critic → PPO → bandits, you have **implemented from scratch** and run the full RL storyline, building core intuitions — value vs policy, model-based vs model-free, on- vs off-policy, exploration vs exploitation — while honestly confronting each algorithm's real behavior.
